Summary: on benchmark le dataloader

In [1]:
from retinotopy import *
welcome()

ModuleNotFoundError: No module named 'cv2'

# Loading legacy images

In [2]:
args = Params()
data_set_type = 'full'
args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images
args.folders = ['train', 'val'] # type of images to use
args

NameError: name 'Params' is not defined

In [3]:
%%timeit -n1
args.folders = ['train', 'val'] # type of images to use
dataloaders = datasets_transforms(args)
len(dataloaders['train']), len(dataloaders['train'].dataset)

NameError: name 'args' is not defined

In [4]:
%%timeit -n1
args.folders = ['val'] # type of images to use
dataloaders = datasets_transforms(args)
len(dataloaders['val']), len(dataloaders['val'].dataset)

NameError: name 'args' is not defined

Benchmarking different methods for the dataloader:

In [5]:
for num_workers_ in [0, 1, 2, 5, 8]: # , 16 , 32
    for batch_size_ in [1, 4, 16, 32, 128, 256, 512]:
        for pin_memory_ in [True, False]:
            args = Params()
            args.batch_size = batch_size_
            args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images
            args.folders = ['val'] # type of images to use            
        
            dataloaders = datasets_transforms(args, pin_memory=pin_memory_, num_workers=num_workers_, verbose=False)
            tic = time.time()
            i_image, i_image_max = 0, 4096
            for i_step, (images, labels) in enumerate(dataloaders['val']):
                    images, labels = images.to(device), labels.to(device)
                    i_image += len(images)
                    if i_image > i_image_max:
                        break

            toc = time.time()
            print(f'{pin_memory_=} \t {num_workers_=} \t {batch_size_=:04d} \t Loading time for {i_image_max} images \t {toc-tic:.1f} s')  

NameError: name 'Params' is not defined

In [6]:
model_filename = f'cached_data/{datetag}_full_resnet101_retino.pt'
model = load_model(model_name='resnet101', model_path=model_filename, do_scratch=False, do_circular=False, verbose=True).to(device)

N_test = 2**8
for num_workers_ in [0, 1, 2, 5, 8]: # , 16 , 32
    for batch_size_ in [1, 4, 16, 32, 64, 128, 256, 512]:
        for pin_memory_ in [True, False]: # [False]: #
            args = Params()
            args.batch_size_val = batch_size_
            args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images
            args.folders = ['val'] # type of images to use            
        
            dataloaders = datasets_transforms(args, pin_memory=pin_memory_, num_workers=num_workers_, verbose=False)
            tic = time.time()
            for i_step, (images, labels) in enumerate(dataloaders['val']):
                    images, labels = images.to(device), labels.to(device)
                    with torch.no_grad():
                        outputs = model(images)
                    if i_step > N_test/batch_size_: break
            toc = time.time()
            print(f'{pin_memory_=} \t\t {num_workers_=} \t\t {batch_size_=:03d} \t\t Elapsed time per image: {1000*(toc-tic)/N_test:.1f} ms')  

NameError: name 'datetag' is not defined